# AMADS Coding Notebooks
## Metrical Position

---

**By (author/s):** Mark Gotham

**For:** Attached to the
[AMADS code library](https://github.com/music-computing/amads/) and
["Keeping Score" book](https://doi.org/10.5281/zenodo.14938027),
but open to all.

**Licence:** MIT.

**Colour key:**
- <font color='green'> Green is for a block of information.
- <font color='purple'> Purple is for an exercise.
- <font color='crimson'> Crimson is for the solution to the exercise above it.

**Learning Objectives:** After completing this notebook, you will be able to ...
- Explore symbolic data.
- Focus on the example use case of note timing information.

## <font color='purple'> Exercise: histogram

- Import a score,
- Retrieve note starts positions relative to the start of the measure,
- Deduce the tatum (and optionally convert to indices representation of events on the tatum grid),
- Plot a histogram of these measure-relative note positions.

In [ ]:
from amads.io.readscore import read_score
from amads.core.basics import Note, Measure
from amads.time.meter.tatum import starts_to_indices, get_tatum_from_priorities
from collections import Counter

from amads.io.readscore import set_reader_warning_level

In [ ]:
set_reader_warning_level("none")

In [ ]:
lieder_base_url = "https://github.com/OpenScore/Lieder/raw/refs/heads/main/scores/"
url = lieder_base_url + "Schubert,_Franz/Op.52/6_Ellens_Gesang_III,_D.839_(Ave_Maria)/lc6389103.mxl"
score_from_url = read_score(url)  # Add the `show=True` argument if you want to see it

In [ ]:
def measure_relative_onsets(
    score,
) -> tuple:
    """
    Your docs here
    """
    pass # Your code here

## <font color='crimson'> Solution

In [ ]:
from typing import NamedTuple

class MeasureOnsets(NamedTuple):
    duration: float
    onsets: list[float]

def measure_relative_onsets(
        score,
        exclude_anomalous: bool = True,
) -> tuple:
    """
    Return measure-relative onsets for all notes in a measured score:
    the time in quarter values since the start of the containing measure.

    Raises
    ------
    Fully measured score is required:
        any score without measures (e.g., flattened score) raises an error.
    If the most-commonly used measure duration is somehow negative.

    Returns
    -------
    Tuple with the measure duration
    (checked to operational throughout)
    and the measure-relative onsets.
    """
    results = []
    measures = [x for x in score.find_all(Measure)]
    if len(measures) == 0:
        raise ValueError("No measures found in score")
    else:
        print(f"{len(measures)} measures found, processing.")
    durations = Counter(measure.duration for measure in measures).most_common()
    reference_duration = durations[0][0]
    if reference_duration < 0:
        raise ValueError(
            f"The most commonly used measure duration ({reference_duration}) is inexplicably negative ... "
        )

    for measure in measures:
        if measure.duration != reference_duration:
            message = f"Anomalous duration of {measure.duration} found."
            if exclude_anomalous:
                print(message + " Skipping anomalous measure.")
                continue
            else:
                raise ValueError(message + " Stopping.")
        for note in measure.find_all(Note):
            results.append(note.onset - measure.onset)

    return MeasureOnsets(duration=reference_duration, onsets=results)

In [ ]:
mrel_onsets = measure_relative_onsets(score_from_url)

This produces a `MeasureOnsets` object with two attributes:
- `mrel_onsets.duration` for the measure duration
- `mrel_onsets.onsets` for the measure-relative onsets

In [ ]:
mrel_onsets.duration  # measure duration

In [ ]:
mrel_onsets.onsets  # measure-relative onsets

In [ ]:
tatum = get_tatum_from_priorities(mrel_onsets.onsets)
tatum

Converting these start position to a list of indices on the tatum.

Note that in this case, we could use the explicit `tatum` argument:

```starts_to_indices(mrel_onsets.onsets, tatum=tatum)```

Then again, when the `tatum` argument is not provided,
`starts_to_indices` calculates a bst guess internally exactly as above, using
`get_tatum_from_priorities`.

In short, the argument is not needed.

In [ ]:
indices = starts_to_indices(mrel_onsets.onsets)  # , tatum=tatum)
indices

In [ ]:
indices_counter = Counter(indices)
indices_counter

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from fractions import Fraction

def fraction_labels(
        beats: int = 4,
        tatum_per_beat: int = 24,
        max_denom_for_ticks: int = 6,
        map_to_1_index: bool = True,
) -> list:
    """
    Simple x labels for measure plotting, minimal redundancy.
    """
    n = beats * tatum_per_beat
    labels = []
    for i in range(n):
        f = Fraction(i * beats, n)
        if f.denominator == 1:   # "0", "1", "2" etc.
            if map_to_1_index:
                labels.append(str(f + 1))
            else:
                labels.append(str(f))
        else:
            if f.denominator > max_denom_for_ticks:
                labels.append("")
            else:
                remainder = f % 1
                labels.append(str(remainder))

    return labels

def plot_metric_histogram(
    c: Counter[float | int],
    beats: int,
    tatum_per_beat: int,
    title: str | None = None,
    norm: bool = False,
    save_path: str | None = None,
    ax=None,
):
    """Plot a histogram of measure-relative onset positions.

    Parameters
    ----------
    c : Counter[float | int]
        A counter for key:values pars for position:count.
    beats : int
        Number of beats in the measure.
    tatum_per_beat : int
        Number of tatum values per beat (assumed to be regular).
    title : str
        Plot title.
    norm : bool
        If True, display proportions instead of raw counts.
    save_path : str
        If no None, use the `save_path` to
    ax : matplotlib Axes, optional
        Draw into an existing Axes; creates a new figure if None.

    Returns
    -------
    fig, ax
    """
    n = beats * tatum_per_beat
    if not c:
        raise ValueError("No data provided.")
    if max(c) > n:
        raise ValueError(
            f"Max value in the counter ({max(c)}) is greater than the number of expected positions {n}."
        )

    xs = range(n)
    counts = np.array(
        [c.get(i, 0) for i in range(n)]
    )

    if norm:
        total = counts.sum()
        if total > 0:
            counts = counts / total
        y_label = "Proportion"
    else:
        y_label = "Note count"

    own_fig = ax is None

    if own_fig:
        fig, ax = plt.subplots(figsize=(8, 4))
    else:
        fig = ax.figure

    fig.patch.set_facecolor("#0f0f14")
    ax.set_facecolor("#0f0f14")

    norm_vals = counts / counts.max() if counts.max() > 0 else counts
    colors = plt.cm.plasma(0.25 + 0.65 * norm_vals)

    width = 0.95
    ax.bar(xs, counts, width=width, color=colors, linewidth=0, zorder=3)

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors="#aaaaaa", labelsize=9)
    ax.xaxis.set_tick_params(length=0)
    ax.yaxis.set_tick_params(length=0)

    # Make the tick labels first ...
    x_ticks = fraction_labels(beats=beats, tatum_per_beat=tatum_per_beat)

    # ... put them in (and gridlines too)
    labeled_positions = [i for i, label in enumerate(x_ticks) if label != ""]
    ax.set_xticks(labeled_positions, [x_ticks[i] for i in labeled_positions])

    for pos in labeled_positions:
        ax.axvline(pos, color="#ffffff18", linewidth=0.1, zorder=0)

    # ... then get them back and format accordingly
    for x in ax.get_xticklabels():
        if len(x.get_text()) > 1:
            x.set_rotation(90)
        else:
            x.set_fontsize(14)

    if norm:
        ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))

    label_color = "#cccccc" if own_fig else "#111111"

    ax.set_xlabel("Beat position", color=label_color, fontsize=10, labelpad=8)
    ax.set_ylabel(y_label, color=label_color, fontsize=10, labelpad=8)
    if title is None:
        title = (
            "Beat Position Usage Profile: "
            f"{beats} beats; "
            f"{tatum_per_beat} tatums per beat; "
            f"{n} tatum bins in total"
        )

    ax.set_title(title, color=label_color, fontsize=13, fontweight="bold", pad=14)
    ax.set_xlim(-width, n - 1 + width)
    ax.set_ylim(0, counts.max() * 1.12)

    fig.tight_layout()

    if own_fig:
        if save_path is not None:
            plt.savefig(save_path, bbox_inches="tight", transparent=True)
        plt.show()

    return fig, ax

In [ ]:
a = plot_metric_histogram(indices_counter, beats=4, tatum_per_beat=24, title="")

In [ ]:
def combined(path_to_score: str, beats: int = 4, beat_quarter_length: float = 1):
    score = read_score(path_to_score)
    data = measure_relative_onsets(score)
    tatum_quarter_length = get_tatum_from_priorities(data.onsets)
    indices = starts_to_indices(data.onsets, tatum=tatum_quarter_length)
    indices_counter = Counter(indices)
    if data.duration != beats * beat_quarter_length:
        print(
            f"warning: the given number of "
            f"beats ({beats}) and beat quarter length ({beat_quarter_length}) "
            f"do not match the main cycle duration ({data.duration}).")
    plot_metric_histogram(
        indices_counter,
        beats=beats,
        tatum_per_beat=int(tatum_quarter_length.denominator * beat_quarter_length),
        title=""
    )

In [ ]:
combined(url)  # Exactly as above

This next cell demonstrates an example generalising beyond 4/4 (here in 12/8).

In [ ]:
combined(
    lieder_base_url + "Schumann%2C_Clara/_/Lorelei/lc4919673.mxl",
    beats=4,
    beat_quarter_length=1.5,
)

Finally, here is an alternative with the in-built histogram functionality.

That function would be over doing it for this use case, but is good to compare.

In [ ]:
from amads.core.histogram import Histogram1D

def counter_to_histogram(counter: Counter, n_bins: int) -> Histogram1D:
    """Convert a Counter of float keys to a Histogram1D with n_bins bins.

    Bin centers are evenly spaced across [min_key, max_key], with boundaries
    at midpoints between centres. The first and last bins are open-ended so
    no Counter entries are ever lost. Counter values are used as weights.
    """
    if not counter:
        raise ValueError("Counter is empty")
    if n_bins < 1:
        raise ValueError("n_bins must be >= 1")

    lo = min(counter)
    hi = max(counter)

    if lo == hi:
        centers = [lo + i * 1.0 for i in range(n_bins)]
    else:
        step = (hi - lo) / (n_bins - 1)
        centers = [lo + i * step for i in range(n_bins)]

    hist = Histogram1D(bin_centers=centers)
    for value, count in counter.items():
        hist.add_point(value, weight=count)
    return hist


End

---